In [1]:
%pip install -q transformers datasets accelerate einops

# Mamba bản ổn định KHÔNG cần build causal-conv1d
%pip install -q mamba-ssm --no-build-isolation

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import copy
import gc
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from mamba_ssm import Mamba


# ===== Device setup =====
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True   # ✅ fix typo here


# ===== Optional: seed for reproducibility =====
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)


# ===== Optional: simple GPU memory cleaner =====
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Setup done ✅")

DEVICE: cuda
Setup done ✅


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

# 👉 FIX: đúng tên dataset
raw_dataset = load_dataset("imdb")

# 👉 tokenizer chung cho 3 model (fair)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
num_labels = raw_dataset["train"].features["label"].num_classes


def tokenize_batch(batch, max_len):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",   # giữ để fair giữa các model
        max_length=max_len,
    )


def build_split(split_name, max_len, n_samples, seed=42):
    dataset = raw_dataset[split_name].shuffle(seed=seed)

    n_samples = min(n_samples, len(dataset))
    dataset = dataset.select(range(n_samples))

    # 👉 FIX: cache để không tokenize lại (rất quan trọng khi benchmark)
    dataset = dataset.map(
        lambda batch: tokenize_batch(batch, max_len),
        batched=True,
        load_from_cache_file=True,
    )

    dataset = dataset.rename_column("label", "labels")

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

    return dataset


def label_distribution(dataset):
    counts = {label_idx: 0 for label_idx in range(num_labels)}
    labels = dataset["labels"]

    for label in labels:
        counts[int(label)] += 1

    total = max(len(labels), 1)
    return {label_idx: round(count / total, 3) for label_idx, count in counts.items()}


def get_loader_from_dataset(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=(DEVICE == "cuda"),
        num_workers=2,  # 👉 thêm để load nhanh hơn
    )


def get_eval_loader(max_len, batch_size, split="test", n_samples=512, seed=42):
    dataset = build_split(split, max_len, n_samples, seed=seed)

    loader = get_loader_from_dataset(
        dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return loader, dataset


def get_train_val_loaders(max_len, batch_size, n_samples=2048, val_ratio=0.2, seed=42):
    dataset = build_split("train", max_len, n_samples, seed=seed)

    # 👉 giữ stratified split (rất đúng)
    split_dataset = dataset.train_test_split(
        test_size=val_ratio,
        seed=seed,
        stratify_by_column="labels",
    )

    train_dataset = split_dataset["train"]
    val_dataset = split_dataset["test"]

    train_loader = get_loader_from_dataset(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )

    val_loader = get_loader_from_dataset(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_loader, val_loader, train_dataset, val_dataset

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DebertaV2Model
from mamba_ssm import Mamba


# =======================
# 1. MAMBA CLASSIFIER
# =======================

class MambaClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        n_layers=4,
        num_labels=4,
        padding_idx=0,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=padding_idx,
        )

        self.layers = nn.ModuleList([
            Mamba(d_model=d_model)
            for _ in range(n_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(x.dtype)
            x = x * mask

        x = self.dropout(x)

        for layer in self.layers:
            x = x + layer(x)  # residual connection

        x = self.norm(x)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(x.dtype)

            x = (
                (x * mask).sum(dim=1)
                / mask.sum(dim=1).clamp(min=1e-6)
            )
        else:
            x = x.mean(dim=1)

        return self.classifier(x)


# =======================
# 2. PATCHTST CLASSIFIER
# =======================

class PatchTSTTextClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        patch_len=16,
        stride=8,
        n_layers=4,
        n_heads=8,
        num_labels=4,
        padding_idx=0,
        dropout=0.1,
        max_patches=512,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=padding_idx,
        )

        self.patch_len = patch_len
        self.stride = stride

        self.patch_proj = nn.Linear(
            d_model * patch_len,
            d_model,
        )

        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        self.positional_encoding = nn.Parameter(
            torch.zeros(1, max_patches + 1, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_labels)

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.positional_encoding, std=0.02)

    def _pad_to_patch(self, embeddings, attention_mask=None):
        seq_len = embeddings.size(1)

        if seq_len <= self.patch_len:
            pad_tokens = self.patch_len - seq_len
        else:
            remainder = (
                (seq_len - self.patch_len)
                % self.stride
            )

            pad_tokens = (
                0
                if remainder == 0
                else self.stride - remainder
            )

        if pad_tokens > 0:
            embeddings = F.pad(
                embeddings,
                (0, 0, 0, pad_tokens),
            )

            if attention_mask is not None:
                attention_mask = F.pad(
                    attention_mask,
                    (0, pad_tokens),
                    value=0,
                )

        return embeddings, attention_mask

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(x.dtype)
            x = x * mask

        x, attention_mask = self._pad_to_patch(
            x,
            attention_mask,
        )

        # Create patches
        patches = x.unfold(
            dimension=1,
            size=self.patch_len,
            step=self.stride,
        ).contiguous()

        B, N, P, D = patches.shape

        patches = patches.view(B, N, P * D)

        patches = self.patch_proj(patches)

        # Add CLS token
        cls_token = self.cls_token.expand(B, -1, -1)

        x = torch.cat(
            [cls_token, patches],
            dim=1,
        )

        # Positional encoding
        x = x + self.positional_encoding[:, :x.size(1)]

        x = self.dropout(x)

        # Attention mask
        key_padding_mask = None

        if attention_mask is not None:
            patch_mask = attention_mask.unfold(
                1,
                self.patch_len,
                self.stride,
            )

            patch_mask = patch_mask.amax(dim=-1).to(torch.bool)

            key_padding_mask = torch.cat(
                [
                    torch.zeros(
                        (patch_mask.size(0), 1),
                        device=x.device,
                        dtype=torch.bool,
                    ),
                    ~patch_mask,
                ],
                dim=1,
            )

        # Transformer encoder
        x = self.encoder(
            x,
            src_key_padding_mask=key_padding_mask,
        )

        # CLS pooling
        x = self.norm(x[:, 0])

        return self.classifier(x)


# =======================
# 3. DEBERTA CLASSIFIER
# =======================

class DebertaClassifier(nn.Module):
    def __init__(
        self,
        model_name="microsoft/deberta-v3-small",
        num_labels=4,
    ):
        super().__init__()

        self.backbone = DebertaV2Model.from_pretrained(
            model_name
        )

        self.classifier = nn.Linear(
            self.backbone.config.hidden_size,
            num_labels,
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls = outputs.last_hidden_state[:, 0]

        return self.classifier(cls)


# =======================
# 4. MODEL FACTORY
# =======================

def build_model(
    model_type,
    vocab_size=None,
    num_labels=4,
):
    if model_type == "mamba":
        return MambaClassifier(
            vocab_size=vocab_size,
            num_labels=num_labels,
        )

    elif model_type == "patchtst":
        return PatchTSTTextClassifier(
            vocab_size=vocab_size,
            num_labels=num_labels,
        )

    elif model_type == "deberta":
        return DebertaClassifier(
            num_labels=num_labels,
        )

    else:
        raise ValueError(
            f"Unknown model_type: {model_type}"
        )

In [5]:
import copy
import torch
import torch.nn as nn
from sklearn.metrics import precision_recall_fscore_support


# =======================
# HELPER FUNCTIONS
# =======================

def extract_logits(outputs):
    return outputs.logits if hasattr(outputs, "logits") else outputs


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# =======================
# EVALUATION
# =======================

def evaluate_loss_and_f1(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    preds = []
    labels = []

    use_amp = DEVICE == "cuda"
    device_type = "cuda" if DEVICE == "cuda" else "cpu"

    with torch.no_grad():
        for batch in loader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch.get("attention_mask", None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(DEVICE)

            y = batch["labels"].to(DEVICE)

            with torch.autocast(
                device_type=device_type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = extract_logits(outputs)

            # 🔥 FIX: luôn cast về float32 để tính loss
            loss = criterion(logits.float(), y)

            total_loss += loss.item()
            total_batches += 1

            pred = torch.argmax(logits.float(), dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    avg_loss = total_loss / max(total_batches, 1)

    f1 = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )[2]

    return avg_loss, f1


# =======================
# TRAINING FUNCTION
# =======================

def train_quick(
    model,
    train_loader,
    val_loader,
    lr=2e-4,
    epochs=10,
    min_epochs=5,
    max_epochs=20,
    patience=4,
):
    model = model.to(DEVICE)
    model = model.float()
    

    # 🔥 FIX: giảm LR nếu là transformer (DeBERTa/BERT)
    if hasattr(model, "backbone") or "deberta" in model.__class__.__name__.lower():
        lr = min(lr, 2e-5)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    epochs = max(min_epochs, min(epochs, max_epochs))

    use_amp = DEVICE == "cuda"
    device_type = "cuda" if DEVICE == "cuda" else "cpu"

    # 🔥 FIX: API mới
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_state = copy.deepcopy(model.state_dict())
    best_val_f1 = -1.0
    best_val_loss = float("inf")
    best_epoch = 0

    no_improve_count = 0
    history = []

    for epoch in range(1, epochs + 1):

        # =======================
        # TRAIN
        # =======================

        model.train()

        train_loss_total = 0.0
        train_batches = 0

        for batch in train_loader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch.get("attention_mask", None)
            if attention_mask is not None:
                attention_mask = attention_mask.to(DEVICE)

            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device_type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = extract_logits(outputs)
                loss = criterion(logits.float(), labels)

            # ❗ FIX: skip NaN
            if torch.isnan(loss):
                continue

            if use_amp:
                scaler.scale(loss).backward()

                # 🔥 FIX QUAN TRỌNG: tránh lỗi FP16 + NaN
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            train_loss_total += loss.item()
            train_batches += 1

        train_loss = train_loss_total / max(train_batches, 1)

        # =======================
        # VALIDATION
        # =======================

        val_loss, val_f1 = evaluate_loss_and_f1(
            model,
            val_loader,
            criterion,
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_f1": val_f1,
            }
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )

        # =======================
        # EARLY STOPPING
        # =======================

        improved = (
            val_f1 > best_val_f1 + 1e-5
        ) or (
            abs(val_f1 - best_val_f1) <= 1e-5
            and val_loss < best_val_loss
        )

        if improved:
            best_state = copy.deepcopy(model.state_dict())
            best_val_f1 = val_f1
            best_val_loss = val_loss
            best_epoch = epoch
            no_improve_count = 0
        else:
            no_improve_count += 1

        if epoch >= min_epochs and no_improve_count >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)

    train_info = {
        "epochs_ran": len(history),
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "best_val_loss": best_val_loss,
        "early_stopped": len(history) < epochs,
        "history": history,
    }

    return model, train_info

In [6]:
def evaluate(model, loader):
    model.eval()

    preds = []
    labels = []

    use_amp = DEVICE == "cuda"

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            y = batch["labels"].to(DEVICE)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = extract_logits(outputs)

            pred = torch.argmax(logits.float(), dim=1)

            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    return acc, precision, recall, f1

In [7]:
def benchmark(model, loader, warmup=5):
    model.eval()

    use_amp = DEVICE == "cuda"

    # 🔥 Warmup
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= warmup:
                break

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = model(input_ids, attention_mask=attention_mask)
                _ = extract_logits(outputs)

    # 🔥 reset memory stats
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    total_tokens = 0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            if DEVICE == "cuda":
                torch.cuda.synchronize()

            # ✅ FIX INDENT + dùng thống nhất autocast
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp
            ):
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = extract_logits(outputs)

            if DEVICE == "cuda":
                torch.cuda.synchronize()

            # 🔥 FAIR token count
            total_tokens += attention_mask.sum().item()
            total_batches += 1

    elapsed = time.time() - start

    memory_mb = (
        torch.cuda.max_memory_allocated() / 1024**2
        if DEVICE == "cuda"
        else 0.0
    )

    return {
        "time": elapsed,
        "throughput_tok_s": total_tokens / max(elapsed, 1e-9),
        "latency_per_batch": elapsed / max(total_batches, 1),
        "memory_mb": memory_mb,
    }

In [8]:
seq_lengths = [256, 512, 1024, 2048]
results = []

TRAIN_EXAMPLES = 2048
TEST_EXAMPLES = 512
VAL_RATIO = 0.2
EPOCHS = 10
MIN_EPOCHS = 5
MAX_EPOCHS = 20
PATIENCE = 4
SEED = 42


def choose_batch_size(seq_len):
    if seq_len <= 256:
        return 32
    if seq_len <= 512:
        return 16
    if seq_len <= 1024:
        return 8
    return 4


def cleanup():
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()


def run_experiment(seq_len):
    print(f"\n🚀 Running seq_len = {seq_len}")

    batch_size = choose_batch_size(seq_len)

    # ===== SAME DATA FOR ALL MODELS =====
    train_loader, val_loader, train_dataset, val_dataset = get_train_val_loaders(
        seq_len,
        batch_size=batch_size,
        n_samples=TRAIN_EXAMPLES,
        val_ratio=VAL_RATIO,
        seed=SEED,
    )

    test_loader, test_dataset = get_eval_loader(
        seq_len,
        batch_size=batch_size,
        split="test",
        n_samples=TEST_EXAMPLES,
        seed=SEED + 1,
    )

    print(
        "label_dist(train/val/test):",
        label_distribution(train_dataset),
        label_distribution(val_dataset),
        label_distribution(test_dataset),
    )

    # =========================================================
    # ======================= MAMBA ============================
    # =========================================================
    cleanup()
    mamba = MambaClassifier(
        vocab_size=tokenizer.vocab_size,
        d_model=256,
        n_layers=4,
        num_labels=num_labels,
        padding_idx=pad_token_id,
    ).to(DEVICE)

    start = time.time()
    mamba, mamba_info = train_quick(
        mamba, train_loader, val_loader,
        lr=2e-4,
        epochs=EPOCHS,
        min_epochs=MIN_EPOCHS,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )
    train_time = time.time() - start

    acc, _, _, f1 = evaluate(mamba, test_loader)
    bench = benchmark(mamba, test_loader)

    print(
        f"Mamba     | acc: {acc:.3f} | f1: {f1:.3f} | "
        f"val_f1*: {mamba_info['best_val_f1']:.3f} | "
        f"epochs: {mamba_info['epochs_ran']} | "
        f"params: {count_parameters(mamba)/1e6:.2f}M | "
        f"train: {train_time:.2f}s | infer: {bench['time']:.2f}s | "
        f"{bench['throughput_tok_s']:.0f} tok/s | {bench['memory_mb']:.0f}MB"
    )

    res_mamba = {
        "acc": acc,
        "f1": f1,
        "params": count_parameters(mamba),
        "train_time": train_time,
        **bench
    }

    del mamba
    cleanup()

    # =========================================================
    # ===================== PATCHTST ===========================
    # =========================================================
    transformer = PatchTSTTextClassifier(
        vocab_size=tokenizer.vocab_size,
        d_model=256,
        patch_len=16,
        stride=8,
        n_layers=4,
        n_heads=8,
        num_labels=num_labels,
        padding_idx=pad_token_id,
    ).to(DEVICE)

    start = time.time()
    transformer, patch_info = train_quick(
        transformer, train_loader, val_loader,
        lr=2e-4,
        epochs=EPOCHS,
        min_epochs=MIN_EPOCHS,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )
    train_time = time.time() - start

    acc, _, _, f1 = evaluate(transformer, test_loader)
    bench = benchmark(transformer, test_loader)

    print(
        f"PatchTST  | acc: {acc:.3f} | f1: {f1:.3f} | "
        f"val_f1*: {patch_info['best_val_f1']:.3f} | "
        f"epochs: {patch_info['epochs_ran']} | "
        f"params: {count_parameters(transformer)/1e6:.2f}M | "
        f"train: {train_time:.2f}s | infer: {bench['time']:.2f}s | "
        f"{bench['throughput_tok_s']:.0f} tok/s | {bench['memory_mb']:.0f}MB"
    )

    res_patch = {
        "acc": acc,
        "f1": f1,
        "params": count_parameters(transformer),
        "train_time": train_time,
        **bench
    }

    del transformer
    cleanup()

    # =========================================================
    # ====================== DEBERTA ===========================
    # =========================================================
    deberta = DebertaClassifier(num_labels=num_labels).to(DEVICE)

    start = time.time()
    deberta, deb_info = train_quick(
        deberta, train_loader, val_loader,
        lr=2e-5,  # 🔥 LR nhỏ hơn (VERY IMPORTANT)
        epochs=EPOCHS,
        min_epochs=MIN_EPOCHS,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )
    train_time = time.time() - start

    acc, _, _, f1 = evaluate(deberta, test_loader)
    bench = benchmark(deberta, test_loader)

    print(
        f"DeBERTa   | acc: {acc:.3f} | f1: {f1:.3f} | "
        f"val_f1*: {deb_info['best_val_f1']:.3f} | "
        f"epochs: {deb_info['epochs_ran']} | "
        f"params: {count_parameters(deberta)/1e6:.2f}M | "
        f"train: {train_time:.2f}s | infer: {bench['time']:.2f}s | "
        f"{bench['throughput_tok_s']:.0f} tok/s | {bench['memory_mb']:.0f}MB"
    )

    res_deb = {
        "acc": acc,
        "f1": f1,
        "params": count_parameters(deberta),
        "train_time": train_time,
        **bench
    }

    del deberta
    cleanup()

    # =========================================================
    # ====================== SAVE ==============================
    # =========================================================
    return {
        "seq_len": seq_len,
        "batch_size": batch_size,
        "mamba": res_mamba,
        "patchtst": res_patch,
        "deberta": res_deb,
    }


# ================= RUN =================
progress_bar = tqdm(seq_lengths, desc="Benchmarking")

for seq_len in progress_bar:
    progress_bar.set_description(f"Processing {seq_len}")
    cleanup()
    results.append(run_experiment(seq_len))

Benchmarking:   0%|          | 0/4 [00:00<?, ?it/s]


🚀 Running seq_len = 256
label_dist(train/val/test): {0: 0.496, 1: 0.504} {0: 0.498, 1: 0.502} {0: 0.543, 1: 0.457}


/tmp/ipykernel_719/1478466630.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Epoch 01 | Train Loss: 0.6937 | Val Loss: 0.6828 | Val F1: 0.5522
Epoch 02 | Train Loss: 0.6688 | Val Loss: 0.6532 | Val F1: 0.6192
Epoch 03 | Train Loss: 0.5813 | Val Loss: nan | Val F1: 0.6767
Epoch 04 | Train Loss: 0.4446 | Val Loss: nan | Val F1: 0.7361
Epoch 05 | Train Loss: 0.4509 | Val Loss: nan | Val F1: 0.6795
Epoch 06 | Train Loss: 0.5276 | Val Loss: nan | Val F1: 0.7202
Epoch 07 | Train Loss: 0.4225 | Val Loss: nan | Val F1: 0.7412
Epoch 08 | Train Loss: 0.3834 | Val Loss: nan | Val F1: 0.7275
Epoch 09 | Train Loss: 0.0000 | Val Loss: nan | Val F1: 0.7275
Epoch 10 | Train Loss: 0.6627 | Val Loss: nan | Val F1: 0.7067
Mamba     | acc: 0.715 | f1: 0.715 | val_f1*: 0.741 | epochs: 10 | params: 9.57M | train: 16.77s | infer: 0.29s | 371789 tok/s | 153MB


/tmp/ipykernel_719/1478466630.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


...

/tmp/ipykernel_719/1478466630.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Epoch 01 | Train Loss: 0.7555 | Val Loss: 0.6422 | Val F1: 0.6169
Epoch 02 | Train Loss: 0.6745 | Val Loss: 0.6618 | Val F1: 0.6207
Epoch 03 | Train Loss: 0.5576 | Val Loss: 0.8192 | Val F1: 0.5788
Epoch 04 | Train Loss: 0.4280 | Val Loss: 1.0794 | Val F1: 0.6900
Epoch 05 | Train Loss: 0.2681 | Val Loss: 1.3267 | Val F1: 0.6340
Epoch 06 | Train Loss: 0.1576 | Val Loss: 1.6175 | Val F1: 0.6732
Epoch 07 | Train Loss: 0.0851 | Val Loss: 1.5889 | Val F1: 0.6976
Epoch 08 | Train Loss: 0.0841 | Val Loss: 1.6857 | Val F1: 0.7065
Epoch 09 | Train Loss: 0.0591 | Val Loss: 2.0014 | Val F1: 0.6798
Epoch 10 | Train Loss: 0.0814 | Val Loss: 2.6614 | Val F1: 0.6484
PatchTST  | acc: 0.674 | f1: 0.674 | val_f1*: 0.706 | epochs: 10 | params: 12.15M | train: 116.38s | infer: 1.21s | 135014 tok/s | 145MB


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_719/1478466630.py:107: FutureWarning:

OutOfMemoryError: CUDA out of memory. Tried to allocate 768.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 661.81 MiB is free. Including non-PyTorch memory, this process has 13.91 GiB memory in use. Of the allocated memory 12.68 GiB is allocated by PyTorch, and 1.10 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print("\n===== FINAL TABLE =====")
print(
    "Seq | Batch | "
    "Mamba_acc | Mamba_f1 | Mamba_train | Mamba_infer | "
    "Patch_acc | Patch_f1 | Patch_train | Patch_infer | "
    "DeBERTa_acc | DeBERTa_f1 | DeBERTa_train | DeBERTa_infer"
)

for r in results:
    print(
        f"{r['seq_len']:>4} | {r['batch_size']:>5} | "
        
        f"{r['mamba']['acc']:.3f} | {r['mamba']['f1']:.3f} | "
        f"{r['mamba']['train_time']:.2f}s | {r['mamba']['time']:.2f}s | "
        
        f"{r['patchtst']['acc']:.3f} | {r['patchtst']['f1']:.3f} | "
        f"{r['patchtst']['train_time']:.2f}s | {r['patchtst']['time']:.2f}s | "
        
        f"{r['deberta']['acc']:.3f} | {r['deberta']['f1']:.3f} | "
        f"{r['deberta']['train_time']:.2f}s | {r['deberta']['time']:.2f}s"
    )